# 97 — Cluster and review nodal stack source positions (SAFE, v3)

Notebook 97 consumes notebook 96's manifests and separates the interpretation
workflow into three explicit stages.

## Stage 1 — Canonical source clustering

Stacks are processed in integration-priority order. Each stack is assigned to
the nearest existing cluster on the same line when its source position is within
the configured tolerance. Otherwise, it starts a new cluster.

The cluster's canonical source position and canonical stack are defined by its
highest-authority member: the lowest numerical `priority`, followed by stable
source-position and stack-ID ordering.

## Stage 2 — Comparison-task generation

Every multi-member cluster generates all unique stack pairs. This is deliberately
branch-agnostic, allowing future `T1_2m`, streamer, or other branches to be added
through notebook 96's branch policy.

## Stage 3 — Waveform evidence

For each task, common receivers are matched one-to-one by position and the
selected component is compared trace by trace using normalized cross-correlation.

This notebook does not merge, overwrite, or resave stack waveforms.

### v3 timing correction

Waveforms are compared on a relative-time axis. Independent stack products are not required to overlap in absolute UTC time.


## 1. Configuration

In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read
from obspy.signal.cross_correlation import correlate, xcorr_max

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
MANIFEST_ROOT = PROJECT_ROOT / '96_unified_nodal_stack_manifest'
OUT_ROOT = PROJECT_ROOT / '97_nodal_source_cluster_review'
FIGURE_ROOT = OUT_ROOT / 'figures'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

STACK_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_manifest.csv'
FILE_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_file_manifest.csv'

SOURCE_TOLERANCE_M = 0.25
RECEIVER_TOLERANCE_M = 0.25
COMPONENT = 'Z'

# Trace comparison.
MAX_LAG_S = 0.100
FILTER_FREQMIN_HZ = None
FILTER_FREQMAX_HZ = None
MIN_COMMON_DURATION_S = 0.10
# Optional relative-time analysis window, measured from each trace start.
# Use None to retain the full common relative duration.
ANALYSIS_START_S = 0.0
ANALYSIS_END_S = None
MIN_COMMON_RECEIVERS_FOR_GATHER = 3

# Screening thresholds only; no automatic merge occurs.
TRACE_ACCEPT_CORRELATION = 0.70
GATHER_ACCEPT_MEDIAN_CORRELATION = 0.70
GATHER_ACCEPT_FRACTION = 0.60
GATHER_MAX_MEDIAN_ABS_LAG_S = 0.010

MAKE_REVIEW_FIGURES = True
MAX_FIGURES = None

pd.set_option('display.max_columns', 250)
pd.set_option('display.width', 260)

print('Manifest root:', MANIFEST_ROOT)
print('Output root:', OUT_ROOT)
print('Source tolerance:', SOURCE_TOLERANCE_M, 'm')
print('Receiver tolerance:', RECEIVER_TOLERANCE_M, 'm')
print('Component:', COMPONENT)

Manifest root: /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest
Output root: /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review
Source tolerance: 0.25 m
Receiver tolerance: 0.25 m
Component: Z


## 2. Load and validate notebook-96 manifests

In [2]:
for path in [STACK_MANIFEST_PATH, FILE_MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f'Missing notebook-96 output: {path}\n'
            'Run notebook 96 first.'
        )

stacks = pd.read_csv(STACK_MANIFEST_PATH, low_memory=False)
files = pd.read_csv(FILE_MANIFEST_PATH, low_memory=False)

required_stack_columns = {
    'stack_id', 'catalog_branch', 'line', 'source_x_m',
    'priority', 'merge_stage',
}
missing_stack_columns = sorted(required_stack_columns - set(stacks.columns))
if missing_stack_columns:
    raise RuntimeError(
        'Notebook-96 stack manifest lacks required columns: '
        f'{missing_stack_columns}'
    )

stacks['source_x_m'] = pd.to_numeric(stacks['source_x_m'], errors='coerce')
stacks['priority'] = pd.to_numeric(stacks['priority'], errors='coerce')

files['component'] = files['component'].astype(str).str.upper()
files['file_type'] = files['file_type'].astype(str).str.lower()
files['file_exists'] = (
    files['file_exists']
    .fillna(False)
    .astype(str)
    .str.lower()
    .isin(['true', '1', 'yes'])
)

invalid = stacks.loc[
    stacks.source_x_m.isna() | stacks.priority.isna(),
    ['stack_id', 'catalog_branch', 'line', 'source_x_m', 'priority'],
]
if len(invalid):
    display(invalid)
    raise RuntimeError(
        'All stacks require finite source_x_m and priority before clustering.'
    )

print('Stack rows:', len(stacks))
print('File rows:', len(files))
display(
    stacks.groupby(
        ['catalog_branch', 'priority', 'merge_stage'],
        dropna=False,
    ).size().reset_index(name='n_stacks')
)

Stack rows: 238
File rows: 978


,catalog_branch,priority,merge_stage,n_stacks
0,geode_linked,1,reference,78
1,geode_linked,2,secondary_geode_extension,36
2,geode_linked,3,streamer_extension,80
3,nodal_only,4,candidate_extension,44


## 3. Assign canonical source clusters

In [3]:
def make_cluster_id(line, number):
    safe_line = ''.join(
        character if character.isalnum() else '_'
        for character in str(line)
    )
    return f'{safe_line}_SRC_{number:04d}'


def choose_canonical_member(member_rows):
    ordered = member_rows.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    )
    return ordered.iloc[0]


cluster_membership_rows = []
cluster_summary_rows = []

for line, line_stacks in stacks.groupby('line', sort=True, dropna=False):
    ordered = line_stacks.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    ).reset_index(drop=True)

    clusters = []

    for stack in ordered.itertuples(index=False):
        possible = []
        for cluster_index, cluster in enumerate(clusters):
            distance = abs(
                float(stack.source_x_m)
                - float(cluster['canonical_source_x_m'])
            )
            if distance <= SOURCE_TOLERANCE_M:
                possible.append(
                    (
                        distance,
                        cluster['canonical_priority'],
                        cluster['canonical_stack_id'],
                        cluster_index,
                    )
                )

        if possible:
            _, _, _, cluster_index = min(possible)
            clusters[cluster_index]['members'].append(stack._asdict())
        else:
            clusters.append({
                'members': [stack._asdict()],
                'canonical_source_x_m': float(stack.source_x_m),
                'canonical_priority': int(stack.priority),
                'canonical_stack_id': str(stack.stack_id),
            })

        # Recompute the canonical member after every assignment. A newly added
        # higher-authority branch can therefore become the canonical member.
        chosen_cluster = clusters[cluster_index] if possible else clusters[-1]
        chosen_frame = pd.DataFrame(chosen_cluster['members'])
        canonical = choose_canonical_member(chosen_frame)
        chosen_cluster['canonical_source_x_m'] = float(canonical.source_x_m)
        chosen_cluster['canonical_priority'] = int(canonical.priority)
        chosen_cluster['canonical_stack_id'] = str(canonical.stack_id)

    clusters.sort(
        key=lambda cluster: (
            cluster['canonical_source_x_m'],
            cluster['canonical_priority'],
            cluster['canonical_stack_id'],
        )
    )

    for cluster_number, cluster in enumerate(clusters, start=1):
        cluster_id = make_cluster_id(line, cluster_number)
        members = pd.DataFrame(cluster['members'])
        canonical = choose_canonical_member(members)

        source_offsets = (
            members.source_x_m.astype(float) - float(canonical.source_x_m)
        )

        cluster_summary_rows.append({
            'source_cluster_id': cluster_id,
            'line': line,
            'canonical_stack_id': str(canonical.stack_id),
            'canonical_catalog_branch': canonical.catalog_branch,
            'canonical_priority': int(canonical.priority),
            'canonical_merge_stage': canonical.merge_stage,
            'canonical_source_x_m': float(canonical.source_x_m),
            'n_stack_products': len(members),
            'n_catalog_branches': members.catalog_branch.nunique(),
            'minimum_source_x_m': float(members.source_x_m.min()),
            'maximum_source_x_m': float(members.source_x_m.max()),
            'source_span_m': float(
                members.source_x_m.max() - members.source_x_m.min()
            ),
            'maximum_abs_offset_from_canonical_m': float(
                source_offsets.abs().max()
            ),
            'is_multi_stack_cluster': len(members) > 1,
            'is_cross_branch_cluster': members.catalog_branch.nunique() > 1,
        })

        for member in members.itertuples(index=False):
            cluster_membership_rows.append({
                'source_cluster_id': cluster_id,
                'line': line,
                'stack_id': str(member.stack_id),
                'catalog_branch': member.catalog_branch,
                'priority': int(member.priority),
                'merge_stage': member.merge_stage,
                'source_x_m': float(member.source_x_m),
                'source_offset_from_canonical_m': (
                    float(member.source_x_m) - float(canonical.source_x_m)
                ),
                'is_canonical_stack': str(member.stack_id) == str(canonical.stack_id),
                'canonical_stack_id': str(canonical.stack_id),
                'canonical_source_x_m': float(canonical.source_x_m),
            })

source_clusters = pd.DataFrame(cluster_summary_rows)
cluster_membership = pd.DataFrame(cluster_membership_rows)

if len(source_clusters):
    source_clusters = source_clusters.sort_values(
        ['line', 'canonical_source_x_m', 'source_cluster_id']
    ).reset_index(drop=True)

if len(cluster_membership):
    cluster_membership = cluster_membership.sort_values(
        ['line', 'canonical_source_x_m', 'priority', 'source_x_m', 'stack_id']
    ).reset_index(drop=True)

print('Source clusters:', len(source_clusters))
print(
    'Multi-stack clusters:',
    int(source_clusters.is_multi_stack_cluster.sum()),
)
print(
    'Cross-branch clusters:',
    int(source_clusters.is_cross_branch_cluster.sum()),
)
display(source_clusters.head(30))

Source clusters: 198
Multi-stack clusters: 37
Cross-branch clusters: 18


,source_cluster_id,line,canonical_stack_id,canonical_catalog_branch,canonical_priority,canonical_merge_stage,canonical_source_x_m,n_stack_products,n_catalog_branches,minimum_source_x_m,maximum_source_x_m,source_span_m,maximum_abs_offset_from_canonical_m,is_multi_stack_cluster,is_cross_branch_cluster
0,T1_SRC_0001,T1,NODALONLYSTACK_MAY19_010M_x0010.0m,nodal_only,4,candidate_extension,10.0,1,1,10.0,10.0,0.0,0.0,False,False
1,T1_SRC_0002,T1,NODALONLYSTACK_MAY19_036M_x0036.0m,nodal_only,4,candidate_extension,36.0,1,1,36.0,36.0,0.0,0.0,False,False
2,T1_SRC_0003,T1,NODALSTACK_T1_T1_2m_refraction_F3047_x0043.0m,geode_linked,2,secondary_geode_extension,43.0,1,1,43.0,43.0,0.0,0.0,False,False
3,T1_SRC_0004,T1,NODALSTACK_T1_T1_2m_refraction_F3048_x0047.0m,geode_linked,2,secondary_geode_extension,47.0,1,1,47.0,47.0,0.0,0.0,False,False
4,T1_SRC_0005,T1,NODALSTACK_T1_T1_2m_refraction_F3049_x0051.0m,geode_linked,2,secondary_geode_extension,51.0,1,1,51.0,51.0,0.0,0.0,False,False
5,T1_SRC_0006,T1,NODALSTACK_T1_T1_2m_refraction_F3050_x0055.0m,geode_linked,2,secondary_geode_extension,55.0,1,1,55.0,55.0,0.0,0.0,False,False
6,T1_SRC_0007,T1,NODALSTACK_T1_T1_2m_refraction_F3051_x0059.0m,geode_linked,2,secondary_geode_extension,59.0,1,1,59.0,59.0,0.0,0.0,False,False
7,T1_SRC_0008,T1,NODALSTACK_T1_T1_2m_refraction_F3052_x0063.0m,geode_linked,2,secondary_geode_extension,63.0,1,1,63.0,63.0,0.0,0.0,False,False
8,T1_SRC_0009,T1,NODALSTACK_T1_T1_2m_refraction_F3053_x0067.0m,geode_linked,2,secondary_geode_extension,67.0,1,1,67.0,67.0,0.0,0.0,False,False
9,T1_SRC_0010,T1,NODALSTACK_T1_T1_2m_refraction_F3054_x0071.0m,geode_linked,2,secondary_geode_extension,71.0,1,1,71.0,71.0,0.0,0.0,False,False


## 4. Validate cluster geometry

In [4]:
cluster_issues = []

too_far = cluster_membership.loc[
    cluster_membership.source_offset_from_canonical_m.abs()
    > SOURCE_TOLERANCE_M + 1e-12
]
if len(too_far):
    cluster_issues.append(
        f'{len(too_far)} members exceed the source tolerance from their canonical source.'
    )

duplicate_membership = cluster_membership.loc[
    cluster_membership.stack_id.duplicated(keep=False)
]
if len(duplicate_membership):
    cluster_issues.append(
        f'{duplicate_membership.stack_id.nunique()} stacks appear in multiple clusters.'
    )

bad_canonical_count = (
    cluster_membership.groupby('source_cluster_id')
    .is_canonical_stack.sum()
)
bad_canonical_count = bad_canonical_count.loc[bad_canonical_count.ne(1)]
if len(bad_canonical_count):
    cluster_issues.append(
        f'{len(bad_canonical_count)} clusters do not have exactly one canonical stack.'
    )

print('Cluster integrity issues:', len(cluster_issues))
for item in cluster_issues:
    print(' -', item)

if len(too_far):
    display(too_far)
if len(duplicate_membership):
    display(duplicate_membership)
if len(bad_canonical_count):
    display(bad_canonical_count.reset_index(name='n_canonical_stacks'))

if cluster_issues:
    raise RuntimeError('Source-cluster integrity checks failed.')
else:
    print('PASS: source clusters satisfy the configured geometry rules.')

Cluster integrity issues: 0
PASS: source clusters satisfy the configured geometry rules.


## 5. Generate all pairwise comparison tasks within clusters

In [5]:
task_rows = []

for cluster_id, members in cluster_membership.groupby(
    'source_cluster_id', sort=False
):
    if len(members) < 2:
        continue

    members = members.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    ).reset_index(drop=True)

    for left_index, right_index in combinations(range(len(members)), 2):
        left = members.iloc[left_index]
        right = members.iloc[right_index]

        if left.priority < right.priority:
            preferred_reference = left
        elif right.priority < left.priority:
            preferred_reference = right
        else:
            preferred_reference = (
                left if (left.source_x_m, left.stack_id)
                <= (right.source_x_m, right.stack_id)
                else right
            )

        task_rows.append({
            'comparison_id': (
                f"{cluster_id}__{left.stack_id}__VS__{right.stack_id}"
            ),
            'source_cluster_id': cluster_id,
            'line': left.line,
            'canonical_stack_id': left.canonical_stack_id,
            'canonical_source_x_m': left.canonical_source_x_m,
            'left_stack_id': left.stack_id,
            'left_catalog_branch': left.catalog_branch,
            'left_priority': int(left.priority),
            'left_merge_stage': left.merge_stage,
            'left_source_x_m': float(left.source_x_m),
            'right_stack_id': right.stack_id,
            'right_catalog_branch': right.catalog_branch,
            'right_priority': int(right.priority),
            'right_merge_stage': right.merge_stage,
            'right_source_x_m': float(right.source_x_m),
            'source_distance_m': abs(
                float(left.source_x_m) - float(right.source_x_m)
            ),
            'is_cross_branch_comparison': (
                left.catalog_branch != right.catalog_branch
            ),
            'preferred_reference_stack_id': preferred_reference.stack_id,
            'preferred_reference_priority': int(preferred_reference.priority),
        })

comparison_tasks = pd.DataFrame(task_rows)

if len(comparison_tasks):
    comparison_tasks = comparison_tasks.sort_values(
        [
            'line', 'canonical_source_x_m', 'source_cluster_id',
            'left_priority', 'right_priority',
            'left_stack_id', 'right_stack_id',
        ]
    ).reset_index(drop=True)

print('Comparison tasks:', len(comparison_tasks))
if len(comparison_tasks):
    display(
        comparison_tasks.groupby(
            [
                'left_catalog_branch', 'right_catalog_branch',
                'is_cross_branch_comparison',
            ],
            dropna=False,
        ).size().reset_index(name='n_tasks')
    )
    display(comparison_tasks.head(30))
else:
    print('No source cluster contains more than one stack product.')

Comparison tasks: 43


,left_catalog_branch,right_catalog_branch,is_cross_branch_comparison,n_tasks
0,geode_linked,geode_linked,False,19
1,geode_linked,nodal_only,True,21
2,nodal_only,nodal_only,False,3


,comparison_id,source_cluster_id,line,canonical_stack_id,canonical_source_x_m,left_stack_id,left_catalog_branch,left_priority,left_merge_stage,left_source_x_m,right_stack_id,right_catalog_branch,right_priority,right_merge_stage,right_source_x_m,source_distance_m,is_cross_branch_comparison,preferred_reference_stack_id,preferred_reference_priority
0,T1_SRC_0022__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0022,T1,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,94.5,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,geode_linked,1,reference,94.5,NODALSTACK_T1_T1_streamer_masw_F1006_x0094.5m,geode_linked,3,streamer_extension,94.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,1
1,T1_SRC_0028__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0028,T1,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,99.0,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,geode_linked,2,secondary_geode_extension,99.0,NODALSTACK_T1_T1_streamer_masw_F1009_x0099.0m,geode_linked,3,streamer_extension,99.0,0.0,False,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,2
2,T1_SRC_0029__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0029,T1,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,100.5,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,geode_linked,1,reference,100.5,NODALSTACK_T1_T1_streamer_masw_F1010_x0100.5m,geode_linked,3,streamer_extension,100.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,1
3,T1_SRC_0036__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0036,T1,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,105.0,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,geode_linked,3,streamer_extension,105.0,NODALONLYSTACK_MAY17_105M_x0105.0m,nodal_only,4,candidate_extension,105.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,3
4,T1_SRC_0038__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0038,T1,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,106.5,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,geode_linked,1,reference,106.5,NODALSTACK_T1_T1_streamer_masw_F1014_x0106.5m,geode_linked,3,streamer_extension,106.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,1
5,T1_SRC_0039__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0039,T1,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,107.0,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,geode_linked,2,secondary_geode_extension,107.0,NODALONLYSTACK_MAY17_107M_x0107.0m,nodal_only,4,candidate_extension,107.0,0.0,True,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,2
6,T1_SRC_0040__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0040,T1,NODALSTACK_T1_T1_streamer_masw_F1015_x0108.0m,108.0,NODALSTACK_T1_T1_streamer_masw_F1015_x0108.0m,geode_linked,3,streamer_extension,108.0,NODALONLYSTACK_MAY17_108M_x0108.0m,nodal_only,4,candidate_extension,108.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1015_x0108.0m,3
7,T1_SRC_0046__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0046,T1,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,111.0,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,geode_linked,2,secondary_geode_extension,111.0,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,geode_linked,3,streamer_extension,111.0,0.0,False,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,2
8,T1_SRC_0046__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0046,T1,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,111.0,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,geode_linked,2,secondary_geode_extension,111.0,NODALONLYSTACK_MAY17_111M_x0111.0m,nodal_only,4,candidate_extension,111.0,0.0,True,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,2
9,T1_SRC_0046__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0046,T1,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,111.0,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,geode_linked,3,streamer_extension,111.0,NODALONLYSTACK_MAY17_111M_x0111.0m,nodal_only,4,candidate_extension,111.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,3


## 6. Resolve waveform products

In [6]:
def resolve_mseed_path(stack_id, component):
    rows = files.loc[
        files.stack_id.astype(str).eq(str(stack_id))
        & files.component.eq(component)
        & files.file_type.eq('mseed')
        & files.file_exists
    ].copy()

    if rows.empty:
        return None

    if len(rows) > 1:
        rows = rows.sort_values('file_path', kind='stable')

    return str(rows.iloc[0].file_path)


if len(comparison_tasks):
    comparison_tasks['left_mseed_path'] = comparison_tasks[
        'left_stack_id'
    ].map(lambda stack_id: resolve_mseed_path(stack_id, COMPONENT))

    comparison_tasks['right_mseed_path'] = comparison_tasks[
        'right_stack_id'
    ].map(lambda stack_id: resolve_mseed_path(stack_id, COMPONENT))

    comparison_tasks['waveforms_available'] = (
        comparison_tasks.left_mseed_path.notna()
        & comparison_tasks.right_mseed_path.notna()
    )

    print(
        'Tasks with both waveform files:',
        int(comparison_tasks.waveforms_available.sum()),
        '/',
        len(comparison_tasks),
    )

    unavailable = comparison_tasks.loc[
        ~comparison_tasks.waveforms_available
    ]
    if len(unavailable):
        display(unavailable[
            ['comparison_id', 'left_mseed_path', 'right_mseed_path']
        ])

Tasks with both waveform files: 43 / 43


## 7. Waveform and receiver-matching helpers

In [7]:
def normalize_component(value):
    text = str(value).strip().upper()
    return text[-1] if text and text[-1] in 'ZNE' else text


def receiver_x_m(trace):
    value = pd.to_numeric(
        getattr(trace.stats, 'receiver_x_m', np.nan),
        errors='coerce',
    )
    if pd.notna(value):
        return float(value)

    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def prepare_stream(path, component):
    stream = read(str(path))
    prepared = []

    for trace in stream:
        if normalize_component(trace.stats.channel) != component:
            continue
        x = receiver_x_m(trace)
        if np.isfinite(x):
            prepared.append((x, trace.copy()))

    return sorted(prepared, key=lambda item: item[0])


def one_to_one_receiver_matches(left, right, tolerance_m):
    candidates = []
    for left_index, (left_x, _) in enumerate(left):
        for right_index, (right_x, _) in enumerate(right):
            distance = abs(float(left_x) - float(right_x))
            if distance <= tolerance_m:
                candidates.append(
                    (distance, left_index, right_index)
                )

    candidates.sort()
    used_left = set()
    used_right = set()
    matches = []

    for distance, left_index, right_index in candidates:
        if left_index in used_left or right_index in used_right:
            continue
        used_left.add(left_index)
        used_right.add(right_index)
        matches.append((left_index, right_index, distance))

    return matches


def preprocess_trace(trace):
    out = trace.copy()
    out.detrend('demean')
    out.detrend('linear')
    out.taper(max_percentage=0.02, type='cosine')

    if FILTER_FREQMIN_HZ is not None and FILTER_FREQMAX_HZ is not None:
        out.filter(
            'bandpass',
            freqmin=FILTER_FREQMIN_HZ,
            freqmax=FILTER_FREQMAX_HZ,
            corners=4,
            zerophase=True,
        )
    elif FILTER_FREQMIN_HZ is not None:
        out.filter(
            'highpass',
            freq=FILTER_FREQMIN_HZ,
            corners=4,
            zerophase=True,
        )
    elif FILTER_FREQMAX_HZ is not None:
        out.filter(
            'lowpass',
            freq=FILTER_FREQMAX_HZ,
            corners=4,
            zerophase=True,
        )

    return out


def trace_relative_window(trace, start_s=0.0, end_s=None):
    """Trim a trace using seconds relative to its own start time."""
    out = trace.copy()
    original_start = out.stats.starttime
    original_end = out.stats.endtime

    start_s = max(0.0, float(start_s))
    end_s = float(end_s) if end_s is not None else float(out.stats.endtime - out.stats.starttime)

    if end_s <= start_s:
        raise ValueError('Relative analysis-window end must exceed start.')

    out.trim(
        original_start + start_s,
        min(original_start + end_s, original_end),
        nearest_sample=True,
    )
    return out


def compare_trace_pair(left_trace, right_trace):
    left_original_start = left_trace.stats.starttime
    right_original_start = right_trace.stats.starttime

    left = preprocess_trace(
        trace_relative_window(
            left_trace,
            start_s=ANALYSIS_START_S,
            end_s=ANALYSIS_END_S,
        )
    )
    right = preprocess_trace(
        trace_relative_window(
            right_trace,
            start_s=ANALYSIS_START_S,
            end_s=ANALYSIS_END_S,
        )
    )

    target_rate = min(
        float(left.stats.sampling_rate),
        float(right.stats.sampling_rate),
    )

    if not np.isclose(left.stats.sampling_rate, target_rate):
        left.resample(target_rate)
    if not np.isclose(right.stats.sampling_rate, target_rate):
        right.resample(target_rate)

    # Compare relative time samples, not absolute UTC timestamps.
    npts = min(left.stats.npts, right.stats.npts)
    common_duration = (npts - 1) / target_rate if npts > 0 else 0.0

    if common_duration < MIN_COMMON_DURATION_S:
        return {
            'status': 'insufficient_common_relative_duration',
            'common_duration_s': common_duration,
            'left_original_starttime': str(left_original_start),
            'right_original_starttime': str(right_original_start),
            'absolute_starttime_difference_s': float(
                right_original_start - left_original_start
            ),
        }

    if npts < 3:
        return {
            'status': 'insufficient_samples',
            'common_duration_s': common_duration,
        }

    left_data = np.asarray(left.data[:npts], dtype=float)
    right_data = np.asarray(right.data[:npts], dtype=float)

    finite = np.isfinite(left_data) & np.isfinite(right_data)
    if finite.sum() < 3:
        return {
            'status': 'nonfinite_data',
            'common_duration_s': common_duration,
        }

    left_data = left_data[finite]
    right_data = right_data[finite]
    left_data -= np.mean(left_data)
    right_data -= np.mean(right_data)

    left_std = np.std(left_data)
    right_std = np.std(right_data)
    if left_std == 0 or right_std == 0:
        return {
            'status': 'constant_trace',
            'common_duration_s': common_duration,
        }

    max_shift_samples = max(1, int(round(MAX_LAG_S * target_rate)))
    correlation = correlate(
        left_data / left_std,
        right_data / right_std,
        max_shift_samples,
        demean=False,
        normalize='naive',
    )
    shift_samples, corrcoef = xcorr_max(
        correlation,
        abs_max=False,
    )

    return {
        'status': 'ok',
        'common_duration_s': common_duration,
        'sampling_rate_hz': target_rate,
        'n_samples': int(len(left_data)),
        'lag_samples': int(shift_samples),
        'lag_s': float(shift_samples) / target_rate,
        'corrcoef': float(corrcoef),
        'left_original_starttime': str(left_original_start),
        'right_original_starttime': str(right_original_start),
        'absolute_starttime_difference_s': float(
            right_original_start - left_original_start
        ),
    }


## 8. Compare common receivers for every task

In [8]:
trace_rows = []
pair_stream_cache = {}

for task in comparison_tasks.itertuples(index=False):
    if not task.waveforms_available:
        continue

    try:
        left_stream = prepare_stream(task.left_mseed_path, COMPONENT)
        right_stream = prepare_stream(task.right_mseed_path, COMPONENT)
        matches = one_to_one_receiver_matches(
            left_stream,
            right_stream,
            RECEIVER_TOLERANCE_M,
        )

        pair_stream_cache[task.comparison_id] = (
            left_stream,
            right_stream,
            matches,
        )

        if not matches:
            trace_rows.append({
                'comparison_id': task.comparison_id,
                'source_cluster_id': task.source_cluster_id,
                'line': task.line,
                'left_stack_id': task.left_stack_id,
                'right_stack_id': task.right_stack_id,
                'source_distance_m': task.source_distance_m,
                'component': COMPONENT,
                'status': 'no_common_receivers',
            })
            continue

        for left_index, right_index, receiver_distance in matches:
            left_x, left_trace = left_stream[left_index]
            right_x, right_trace = right_stream[right_index]

            result = compare_trace_pair(left_trace, right_trace)

            trace_rows.append({
                'comparison_id': task.comparison_id,
                'source_cluster_id': task.source_cluster_id,
                'line': task.line,
                'left_stack_id': task.left_stack_id,
                'right_stack_id': task.right_stack_id,
                'source_distance_m': task.source_distance_m,
                'component': COMPONENT,
                'left_receiver_x_m': left_x,
                'right_receiver_x_m': right_x,
                'receiver_distance_m': receiver_distance,
                'left_station': left_trace.stats.station,
                'right_station': right_trace.stats.station,
                **result,
            })

    except Exception as exc:
        trace_rows.append({
            'comparison_id': task.comparison_id,
            'source_cluster_id': task.source_cluster_id,
            'line': task.line,
            'left_stack_id': task.left_stack_id,
            'right_stack_id': task.right_stack_id,
            'source_distance_m': task.source_distance_m,
            'component': COMPONENT,
            'status': 'pair_processing_error',
            'error': repr(exc),
        })

trace_qc = pd.DataFrame(trace_rows)

print('Trace comparison rows:', len(trace_qc))
if len(trace_qc):
    display(
        trace_qc.groupby('status', dropna=False)
        .size().reset_index(name='n_rows')
    )

Trace comparison rows: 1212


,status,n_rows
0,ok,1212


## 9. Summarize pair-level waveform evidence

In [9]:
review_rows = []

for task in comparison_tasks.itertuples(index=False):
    rows = trace_qc.loc[
        trace_qc.comparison_id.eq(task.comparison_id)
        & trace_qc.status.eq('ok')
    ].copy()

    if len(rows):
        passing = rows.corrcoef >= TRACE_ACCEPT_CORRELATION
        n_common = len(rows)
        fraction_passing = float(passing.mean())
        median_corr = float(rows.corrcoef.median())
        minimum_corr = float(rows.corrcoef.min())
        median_lag = float(rows.lag_s.median())
        median_abs_lag = float(rows.lag_s.abs().median())
        lag_mad = float(
            np.median(np.abs(rows.lag_s - np.median(rows.lag_s)))
        )
    else:
        n_common = 0
        fraction_passing = np.nan
        median_corr = np.nan
        minimum_corr = np.nan
        median_lag = np.nan
        median_abs_lag = np.nan
        lag_mad = np.nan

    enough_receivers = n_common >= MIN_COMMON_RECEIVERS_FOR_GATHER
    passes_waveform_thresholds = (
        enough_receivers
        and median_corr >= GATHER_ACCEPT_MEDIAN_CORRELATION
        and fraction_passing >= GATHER_ACCEPT_FRACTION
        and median_abs_lag <= GATHER_MAX_MEDIAN_ABS_LAG_S
    )

    if not task.waveforms_available:
        automatic_status = 'missing_waveform_product'
    elif not enough_receivers:
        automatic_status = 'insufficient_common_receivers'
    elif passes_waveform_thresholds:
        automatic_status = 'waveform_match_supported'
    else:
        automatic_status = 'waveform_match_not_supported'

    row = task._asdict()
    row.update({
        'n_common_receivers': n_common,
        'n_trace_correlations_passing': (
            int((rows.corrcoef >= TRACE_ACCEPT_CORRELATION).sum())
            if len(rows) else 0
        ),
        'fraction_trace_correlations_passing': fraction_passing,
        'median_trace_corrcoef': median_corr,
        'minimum_trace_corrcoef': minimum_corr,
        'median_lag_s': median_lag,
        'median_abs_lag_s': median_abs_lag,
        'lag_mad_s': lag_mad,
        'automatic_status': automatic_status,
        'manual_review_status': 'not_reviewed',
        'same_physical_shot_decision': 'undecided',
        'review_notes': '',
    })
    review_rows.append(row)

pair_review = pd.DataFrame(review_rows)

if len(pair_review):
    pair_review = pair_review.sort_values(
        [
            'line', 'canonical_source_x_m', 'source_cluster_id',
            'left_priority', 'right_priority',
            'left_stack_id', 'right_stack_id',
        ]
    ).reset_index(drop=True)

print('Pair-level review rows:', len(pair_review))
if len(pair_review):
    display(
        pair_review.groupby('automatic_status', dropna=False)
        .size().reset_index(name='n_comparisons')
    )
    display(pair_review.head(30))

Pair-level review rows: 43


,automatic_status,n_comparisons
0,waveform_match_not_supported,41
1,waveform_match_supported,2


,comparison_id,source_cluster_id,line,canonical_stack_id,canonical_source_x_m,left_stack_id,left_catalog_branch,left_priority,left_merge_stage,left_source_x_m,right_stack_id,right_catalog_branch,right_priority,right_merge_stage,right_source_x_m,source_distance_m,is_cross_branch_comparison,preferred_reference_stack_id,preferred_reference_priority,left_mseed_path,right_mseed_path,waveforms_available,n_common_receivers,n_trace_correlations_passing,fraction_trace_correlations_passing,median_trace_corrcoef,minimum_trace_corrcoef,median_lag_s,median_abs_lag_s,lag_mad_s,automatic_status,manual_review_status,same_physical_shot_decision,review_notes
0,T1_SRC_0022__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0022,T1,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,94.5,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,geode_linked,1,reference,94.5,NODALSTACK_T1_T1_streamer_masw_F1006_x0094.5m,geode_linked,3,streamer_extension,94.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,1,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,0,0.000000,0.086936,0.003333,-0.069,0.069,0.019,waveform_match_not_supported,not_reviewed,undecided,
1,T1_SRC_0028__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0028,T1,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,99.0,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,geode_linked,2,secondary_geode_extension,99.0,NODALSTACK_T1_T1_streamer_masw_F1009_x0099.0m,geode_linked,3,streamer_extension,99.0,0.0,False,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,2,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,18,0.692308,0.833648,0.582596,0.014,0.014,0.000,waveform_match_not_supported,not_reviewed,undecided,
2,T1_SRC_0029__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0029,T1,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,100.5,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,geode_linked,1,reference,100.5,NODALSTACK_T1_T1_streamer_masw_F1010_x0100.5m,geode_linked,3,streamer_extension,100.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,1,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,0,0.000000,0.170386,0.020679,0.070,0.070,0.022,waveform_match_not_supported,not_reviewed,undecided,
3,T1_SRC_0036__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0036,T1,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,105.0,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,geode_linked,3,streamer_extension,105.0,NODALONLYSTACK_MAY17_105M_x0105.0m,nodal_only,4,candidate_extension,105.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,3,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,True,26,23,0.884615,0.819313,0.401475,-0.062,0.062,0.004,waveform_match_not_supported,not_reviewed,undecided,
4,T1_SRC_0038__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0038,T1,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,106.5,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,geode_linked,1,reference,106.5,NODALSTACK_T1_T1_streamer_masw_F1014_x0106.5m,geode_linked,3,streamer_extension,106.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,1,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,0,0.000000,0.318459,0.102981,0.073,0.073,0.009,waveform_match_not_supported,not_reviewed,undecided,
5,T1_SRC_0039__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0039,T1,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,107.0,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,geode_linked,2,secondary_geode_extension,107.0,NODALONLYSTACK_MAY17_107M_x0107.0m,nodal_only,4,candidate_extension,107.0,0.0,True,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,2,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,True,33,0,0.000000,0.116987,0.005738,0.074,0.076,0.016,waveform_match_not_supported,not_reviewed,undecided,
6,T1_SRC_0040__NODALSTACK_T1_T1_streamer_masw_F1..

## 10. Build cluster-level review summary

In [10]:
cluster_review = source_clusters.copy()

if len(pair_review):
    pair_counts = (
        pair_review.groupby('source_cluster_id')
        .agg(
            n_comparison_tasks=('comparison_id', 'size'),
            n_waveform_match_supported=(
                'automatic_status',
                lambda series: int(
                    series.eq('waveform_match_supported').sum()
                ),
            ),
            n_waveform_match_not_supported=(
                'automatic_status',
                lambda series: int(
                    series.eq('waveform_match_not_supported').sum()
                ),
            ),
            n_insufficient_common_receivers=(
                'automatic_status',
                lambda series: int(
                    series.eq('insufficient_common_receivers').sum()
                ),
            ),
            best_median_trace_corrcoef=(
                'median_trace_corrcoef', 'max'
            ),
            worst_median_trace_corrcoef=(
                'median_trace_corrcoef', 'min'
            ),
        )
        .reset_index()
    )

    cluster_review = cluster_review.merge(
        pair_counts,
        on='source_cluster_id',
        how='left',
        validate='one_to_one',
    )
else:
    cluster_review['n_comparison_tasks'] = 0

for column in [
    'n_comparison_tasks',
    'n_waveform_match_supported',
    'n_waveform_match_not_supported',
    'n_insufficient_common_receivers',
]:
    if column not in cluster_review:
        cluster_review[column] = 0
    cluster_review[column] = cluster_review[column].fillna(0).astype(int)

cluster_review['manual_cluster_status'] = 'not_reviewed'
cluster_review['canonical_source_decision'] = np.where(
    cluster_review.n_stack_products.eq(1),
    'single_stack_only',
    'undecided',
)
cluster_review['cluster_review_notes'] = ''

display(cluster_review.head(30))

,source_cluster_id,line,canonical_stack_id,canonical_catalog_branch,canonical_priority,canonical_merge_stage,canonical_source_x_m,n_stack_products,n_catalog_branches,minimum_source_x_m,maximum_source_x_m,source_span_m,maximum_abs_offset_from_canonical_m,is_multi_stack_cluster,is_cross_branch_cluster,n_comparison_tasks,n_waveform_match_supported,n_waveform_match_not_supported,n_insufficient_common_receivers,best_median_trace_corrcoef,worst_median_trace_corrcoef,manual_cluster_status,canonical_source_decision,cluster_review_notes
0,T1_SRC_0001,T1,NODALONLYSTACK_MAY19_010M_x0010.0m,nodal_only,4,candidate_extension,10.0,1,1,10.0,10.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
1,T1_SRC_0002,T1,NODALONLYSTACK_MAY19_036M_x0036.0m,nodal_only,4,candidate_extension,36.0,1,1,36.0,36.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
2,T1_SRC_0003,T1,NODALSTACK_T1_T1_2m_refraction_F3047_x0043.0m,geode_linked,2,secondary_geode_extension,43.0,1,1,43.0,43.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
3,T1_SRC_0004,T1,NODALSTACK_T1_T1_2m_refraction_F3048_x0047.0m,geode_linked,2,secondary_geode_extension,47.0,1,1,47.0,47.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
4,T1_SRC_0005,T1,NODALSTACK_T1_T1_2m_refraction_F3049_x0051.0m,geode_linked,2,secondary_geode_extension,51.0,1,1,51.0,51.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
5,T1_SRC_0006,T1,NODALSTACK_T1_T1_2m_refraction_F3050_x0055.0m,geode_linked,2,secondary_geode_extension,55.0,1,1,55.0,55.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
6,T1_SRC_0007,T1,NODALSTACK_T1_T1_2m_refraction_F3051_x0059.0m,geode_linked,2,secondary_geode_extension,59.0,1,1,59.0,59.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
7,T1_SRC_0008,T1,NODALSTACK_T1_T1_2m_refraction_F3052_x0063.0m,geode_linked,2,secondary_geode_extension,63.0,1,1,63.0,63.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
8,T1_SRC_0009,T1,NODALSTACK_T1_T1_2m_refraction_F3053_x0067.0m,geode_linked,2,secondary_geode_extension,67.0,1,1,67.0,67.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
9,T1_SRC_0010,T1,NODALSTACK_T1_T1_2m_refraction_F3054_x0071.0m,geode_linked,2,secondary_geode_extension,71.0,1,1,71.0,71.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,


## 11. Create review figures

In [11]:
def safe_name(value):
    return ''.join(
        character if character.isalnum() or character in '-_.'
        else '_'
        for character in str(value)
    )


figure_paths = []
figure_reviews = pair_review.copy()

if MAX_FIGURES is not None:
    figure_reviews = figure_reviews.head(MAX_FIGURES)

if MAKE_REVIEW_FIGURES:
    for review in figure_reviews.itertuples(index=False):
        rows = trace_qc.loc[
            trace_qc.comparison_id.eq(review.comparison_id)
            & trace_qc.status.eq('ok')
        ].copy()

        if rows.empty or review.comparison_id not in pair_stream_cache:
            continue

        left_stream, right_stream, _ = pair_stream_cache[
            review.comparison_id
        ]

        fig, ax = plt.subplots(figsize=(12, 8))
        vertical_offset = 0.0

        for row in rows.sort_values(
            'left_receiver_x_m'
        ).itertuples(index=False):
            left_item = min(
                left_stream,
                key=lambda item: abs(
                    item[0] - row.left_receiver_x_m
                ),
            )
            right_item = min(
                right_stream,
                key=lambda item: abs(
                    item[0] - row.right_receiver_x_m
                ),
            )

            left_trace = preprocess_trace(left_item[1])
            right_trace = preprocess_trace(right_item[1])

            rate = min(
                float(left_trace.stats.sampling_rate),
                float(right_trace.stats.sampling_rate),
            )
            if not np.isclose(left_trace.stats.sampling_rate, rate):
                left_trace.resample(rate)
            if not np.isclose(right_trace.stats.sampling_rate, rate):
                right_trace.resample(rate)

            left_trace = trace_relative_window(
                left_trace,
                start_s=ANALYSIS_START_S,
                end_s=ANALYSIS_END_S,
            )
            right_trace = trace_relative_window(
                right_trace,
                start_s=ANALYSIS_START_S,
                end_s=ANALYSIS_END_S,
            )

            npts = min(left_trace.stats.npts, right_trace.stats.npts)
            if npts < 3:
                continue

            left_data = np.asarray(
                left_trace.data[:npts], dtype=float
            )
            right_data = np.asarray(
                right_trace.data[:npts], dtype=float
            )

            left_scale = np.nanmax(np.abs(left_data))
            right_scale = np.nanmax(np.abs(right_data))
            if np.isfinite(left_scale) and left_scale > 0:
                left_data = left_data / left_scale
            if np.isfinite(right_scale) and right_scale > 0:
                right_data = right_data / right_scale

            lag_samples = int(round(row.lag_s * rate))
            aligned_right = np.full_like(right_data, np.nan, dtype=float)
            if lag_samples > 0:
                aligned_right[lag_samples:] = right_data[:-lag_samples]
            elif lag_samples < 0:
                aligned_right[:lag_samples] = right_data[-lag_samples:]
            else:
                aligned_right[:] = right_data

            times = np.arange(npts) / rate
            ax.plot(times, left_data + vertical_offset, linewidth=0.8)
            ax.plot(
                times,
                aligned_right + vertical_offset,
                linewidth=0.8,
                alpha=0.75,
            )
            ax.text(
                times[-1] if len(times) else 0,
                vertical_offset,
                (
                    f" x={row.left_receiver_x_m:.2f}/"
                    f"{row.right_receiver_x_m:.2f} m, "
                    f"r={row.corrcoef:.2f}, "
                    f"lag={row.lag_s * 1000:.1f} ms"
                ),
                va='center',
                fontsize=7,
            )
            vertical_offset += 2.5

        ax.set_title(
            f"{review.source_cluster_id}: "
            f"{review.left_stack_id} vs {review.right_stack_id}\n"
            f"source separation={review.source_distance_m:.3f} m; "
            f"median r={review.median_trace_corrcoef:.3f}; "
            f"status={review.automatic_status}"
        )
        ax.set_xlabel('Time from common trace start (s)')
        ax.set_ylabel('Normalized traces with vertical offsets (right trace lag-aligned)')
        ax.grid(True, alpha=0.25)
        fig.tight_layout()

        figure_path = (
            FIGURE_ROOT
            / f"{safe_name(review.comparison_id)}_{COMPONENT}.png"
        )
        fig.savefig(figure_path, dpi=180)
        plt.close(fig)
        figure_paths.append(str(figure_path))

print('Review figures written:', len(figure_paths))

Review figures written: 43


## 12. Export cluster, task, and waveform-review tables

In [12]:
OUTPUTS = {
    'clusters': OUT_ROOT / '97_source_clusters.csv',
    'membership': OUT_ROOT / '97_source_cluster_membership.csv',
    'tasks': OUT_ROOT / '97_source_cluster_comparison_tasks.csv',
    'trace_qc': OUT_ROOT / '97_candidate_trace_correlation_qc.csv',
    'pair_review': OUT_ROOT / '97_candidate_pair_review.csv',
    'cluster_review': OUT_ROOT / '97_source_cluster_review.csv',
    'summary': OUT_ROOT / '97_source_cluster_review_summary.csv',
}

source_clusters.to_csv(OUTPUTS['clusters'], index=False)
cluster_membership.to_csv(OUTPUTS['membership'], index=False)
comparison_tasks.to_csv(OUTPUTS['tasks'], index=False)
trace_qc.to_csv(OUTPUTS['trace_qc'], index=False)
pair_review.to_csv(OUTPUTS['pair_review'], index=False)
cluster_review.to_csv(OUTPUTS['cluster_review'], index=False)

summary = pd.DataFrame([
    ('source_tolerance_m', SOURCE_TOLERANCE_M),
    ('receiver_tolerance_m', RECEIVER_TOLERANCE_M),
    ('component', COMPONENT),
    ('input_stack_products', len(stacks)),
    ('source_clusters', len(source_clusters)),
    ('single_stack_clusters', int(
        source_clusters.n_stack_products.eq(1).sum()
    ) if len(source_clusters) else 0),
    ('multi_stack_clusters', int(
        source_clusters.n_stack_products.gt(1).sum()
    ) if len(source_clusters) else 0),
    ('cross_branch_clusters', int(
        source_clusters.is_cross_branch_cluster.sum()
    ) if len(source_clusters) else 0),
    ('comparison_tasks', len(comparison_tasks)),
    ('tasks_with_waveforms', int(
        comparison_tasks.waveforms_available.sum()
    ) if len(comparison_tasks) else 0),
    ('successful_trace_comparisons', int(
        trace_qc.status.eq('ok').sum()
    ) if len(trace_qc) else 0),
    ('waveform_match_supported', int(
        pair_review.automatic_status.eq(
            'waveform_match_supported'
        ).sum()
    ) if len(pair_review) else 0),
    ('waveform_match_not_supported', int(
        pair_review.automatic_status.eq(
            'waveform_match_not_supported'
        ).sum()
    ) if len(pair_review) else 0),
    ('review_figures', len(figure_paths)),
], columns=['metric', 'value'])

summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for key, path in OUTPUTS.items():
    print(f'  {key:16s} {path}')
print('  figures          ', FIGURE_ROOT)

,metric,value
0,source_tolerance_m,0.25
1,receiver_tolerance_m,0.25
2,component,Z
3,input_stack_products,238
4,source_clusters,198
5,single_stack_clusters,161
6,multi_stack_clusters,37
7,cross_branch_clusters,18
8,comparison_tasks,43
9,tasks_with_waveforms,43



Written:
  clusters         /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_clusters.csv
  membership       /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_membership.csv
  tasks            /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_comparison_tasks.csv
  trace_qc         /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_candidate_trace_correlation_qc.csv
  pair_review      /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_candidate_pair_review.csv
  cluster_review   /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_review.csv
  summary          /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_review_summary.csv
  figures           /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/figures


## 13. Interpretation and later integration

The source cluster is a geometric hypothesis. Waveform evidence may support or
reject individual pairings inside that cluster.

Review:

- `97_candidate_pair_review.csv`
- `97_source_cluster_review.csv`
- the generated figures

and update the manual-decision columns.

A later integration notebook should merge only approved relationships, retain
every original stack identifier, preserve receiver-level provenance, and use the
canonical stack/source defined by the lowest numerical priority.